# 📊 Statistics Education Material

## Part 1: Intermediate Statistics | Part 2: Advanced Statistics

This notebook covers comprehensive statistical concepts with theory, formulas, and code examples.

---
**Contents:**
- **Part 1**: T-test, Z-test, Chi-square, Confidence Intervals, Normality tests, Variance homogeneity
- **Part 2**: Regression analysis, ANOVA, Correlation analysis, Post-hoc tests

## 🔧 Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import anova_lm
import warnings

warnings.filterwarnings('ignore')
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

print('Libraries loaded successfully!')

---
# Part 1: Intermediate Statistics

Intermediate statistical tests help us make inferences about populations from samples.
We test **hypotheses** and quantify the probability of observing results by chance.

## 1.1 T-test

The **t-test** is used to compare means when the population standard deviation is unknown.

### Types:
- **Independent samples t-test**: Compare means of two separate groups
- **Paired samples t-test**: Compare means of the same group under two conditions

### Formula (Independent Samples):
$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

### Assumptions:
1. Data is approximately normally distributed
2. Observations are independent
3. Equal variances (for standard t-test) or unequal variances (Welch's t-test)

In [ ]:
# ============================================================
# 1.1 Independent Samples T-test
# ============================================================

# Generate sample data
group_a = np.random.normal(loc=50, scale=10, size=30)
group_b = np.random.normal(loc=55, scale=10, size=30)

# Perform t-test
t_stat, p_value = stats.ttest_ind(group_a, group_b)

print('Independent Samples T-test')
print(f'  Group A: mean={group_a.mean():.2f}, std={group_a.std():.2f}, n={len(group_a)}')
print(f'  Group B: mean={group_b.mean():.2f}, std={group_b.std():.2f}, n={len(group_b)}')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value: {p_value:.4f}')
print(f'  Result: {"Reject H0 (significant difference)" if p_value < 0.05 else "Fail to reject H0 (no significant difference)"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Independent Samples T-test', fontsize=14, fontweight='bold')

# Histogram
axes[0].hist(group_a, bins=15, alpha=0.6, color='steelblue', label='Group A')
axes[0].hist(group_b, bins=15, alpha=0.6, color='coral', label='Group B')
axes[0].axvline(group_a.mean(), color='steelblue', linestyle='--', linewidth=2)
axes[0].axvline(group_b.mean(), color='coral', linestyle='--', linewidth=2)
axes[0].set_title('Distribution Comparison')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
data_for_box = pd.DataFrame({'Value': np.concatenate([group_a, group_b]),
                              'Group': ['A']*len(group_a) + ['B']*len(group_b)})
data_for_box.boxplot(column='Value', by='Group', ax=axes[1])
axes[1].set_title('Box Plot by Group')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 1.1 Paired Samples T-test
# ============================================================

# Generate paired data (before/after treatment)
before = np.random.normal(loc=70, scale=8, size=25)
after = before + np.random.normal(loc=5, scale=3, size=25)  # effect of +5

# Paired t-test
t_stat_paired, p_value_paired = stats.ttest_rel(before, after)

print('Paired Samples T-test (Before vs After)')
print(f'  Before: mean={before.mean():.2f}, std={before.std():.2f}')
print(f'  After:  mean={after.mean():.2f}, std={after.std():.2f}')
print(f'  Mean difference: {(after - before).mean():.2f}')
print(f'  t-statistic: {t_stat_paired:.4f}')
print(f'  p-value: {p_value_paired:.4f}')
print(f'  Result: {"Reject H0 (significant change)" if p_value_paired < 0.05 else "Fail to reject H0"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Paired Samples T-test', fontsize=14, fontweight='bold')

# Before vs After scatter
axes[0].scatter(before, after, alpha=0.7, color='steelblue', s=50)
axes[0].plot([before.min(), before.max()], [before.min(), before.max()], 'r--', label='No change line')
axes[0].set_title('Before vs After')
axes[0].set_xlabel('Before')
axes[0].set_ylabel('After')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Differences
diff = after - before
axes[1].hist(diff, bins=10, color='coral', alpha=0.7, edgecolor='black')
axes[1].axvline(diff.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean diff: {diff.mean():.2f}')
axes[1].axvline(0, color='black', linestyle='-', linewidth=1, label='Zero')
axes[1].set_title('Distribution of Differences')
axes[1].set_xlabel('After - Before')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.2 Z-test

The **Z-test** is used when the population standard deviation is **known** or the sample size is large (n > 30).

### Formula:
$$z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}}$$

Where:
- $\bar{X}$ = sample mean
- $\mu_0$ = hypothesized population mean
- $\sigma$ = known population standard deviation
- $n$ = sample size

### When to use Z-test vs T-test:
| Condition | Test |
|-----------|------|
| Population σ known | Z-test |
| Population σ unknown, n < 30 | T-test |
| Population σ unknown, n ≥ 30 | Either (Z-test approximation) |

In [ ]:
# ============================================================
# 1.2 One-sample Z-test
# ============================================================

from scipy.stats import norm

# Known population parameters
pop_mean = 100  # hypothesized population mean
pop_std = 15    # known population standard deviation

# Sample data
sample = np.random.normal(loc=105, scale=15, size=50)
n = len(sample)
sample_mean = sample.mean()

# Z-test calculation
z_stat = (sample_mean - pop_mean) / (pop_std / np.sqrt(n))
p_value_z = 2 * (1 - norm.cdf(abs(z_stat)))  # two-tailed

print('One-sample Z-test')
print(f'  Hypothesized mean (H0): {pop_mean}')
print(f'  Population std (known): {pop_std}')
print(f'  Sample mean: {sample_mean:.2f}')
print(f'  Sample size: {n}')
print(f'  Z-statistic: {z_stat:.4f}')
print(f'  p-value (two-tailed): {p_value_z:.4f}')
print(f'  Critical value (alpha=0.05): ±{norm.ppf(0.975):.4f}')
print(f'  Result: {"Reject H0" if p_value_z < 0.05 else "Fail to reject H0"}')

# Visualization: Z-distribution
x = np.linspace(-4, 4, 300)
y = norm.pdf(x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, y, 'b-', linewidth=2, label='Standard Normal')

# Critical regions
critical_val = 1.96
x_left = x[x < -critical_val]
x_right = x[x > critical_val]
ax.fill_between(x_left, norm.pdf(x_left), alpha=0.4, color='red', label='Rejection region (alpha=0.05)')
ax.fill_between(x_right, norm.pdf(x_right), alpha=0.4, color='red')

# Z-statistic
ax.axvline(z_stat, color='green', linestyle='--', linewidth=2, label=f'Z-statistic = {z_stat:.3f}')
ax.axvline(-critical_val, color='red', linestyle=':', linewidth=1.5, label=f'Critical values = ±{critical_val}')
ax.axvline(critical_val, color='red', linestyle=':', linewidth=1.5)

ax.set_title('Z-test: Standard Normal Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Z-score')
ax.set_ylabel('Probability Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 1.3 Chi-square Test ($\chi^2$)

The **Chi-square test** is used for categorical data:
- **Goodness of fit**: Tests if observed frequencies match expected frequencies
- **Test of independence**: Tests if two categorical variables are independent

### Formula:
$$\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}$$

Where:
- $O_i$ = observed frequency
- $E_i$ = expected frequency
- $df = (rows-1)(cols-1)$ for test of independence

In [ ]:
# ============================================================
# 1.3 Chi-square Test of Independence
# ============================================================

# Contingency table: Marketing channel vs Purchase
# Rows: Marketing channel (Email, Social, Search)
# Cols: Purchase (Yes, No)
contingency_table = np.array([
    [45, 55],   # Email
    [70, 30],   # Social Media
    [50, 50],   # Search
])

channel_names = ['Email', 'Social Media', 'Search']
purchase_names = ['Purchase', 'No Purchase']

# Chi-square test
chi2_stat, p_value_chi2, dof, expected = stats.chi2_contingency(contingency_table)

print('Chi-square Test of Independence')
print('\nObserved frequencies:')
print(pd.DataFrame(contingency_table, index=channel_names, columns=purchase_names))
print('\nExpected frequencies:')
print(pd.DataFrame(expected.round(2), index=channel_names, columns=purchase_names))
print(f'\nChi-square statistic: {chi2_stat:.4f}')
print(f'Degrees of freedom: {dof}')
print(f'p-value: {p_value_chi2:.4f}')
print(f'Result: {"Reject H0 (variables are dependent)" if p_value_chi2 < 0.05 else "Fail to reject H0 (variables are independent)"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Chi-square Test of Independence', fontsize=14, fontweight='bold')

# Stacked bar
df_ct = pd.DataFrame(contingency_table, index=channel_names, columns=purchase_names)
df_ct.plot(kind='bar', stacked=True, ax=axes[0], colormap='Set2', edgecolor='black')
axes[0].set_title('Observed Frequencies (Stacked)')
axes[0].set_xlabel('Marketing Channel')
axes[0].set_ylabel('Count')
axes[0].legend(loc='upper right')
axes[0].tick_params(axis='x', rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Heatmap of residuals
residuals = (contingency_table - expected) / np.sqrt(expected)
sns.heatmap(pd.DataFrame(residuals, index=channel_names, columns=purchase_names),
            annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('Pearson Residuals\n(|value| > 2 = significant)')

plt.tight_layout()
plt.show()

## 1.4 Confidence Interval (CI)

A **confidence interval** gives a range of plausible values for a population parameter.

### Formula (for mean, σ unknown):
$$CI = \bar{X} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}}$$

### Interpretation:
A 95% CI means: **"If we repeated the experiment many times, 95% of the intervals would contain the true population mean."**

> ⚠️ Common misconception: It does **NOT** mean 'there is a 95% probability the true mean lies in this interval.'

In [ ]:
# ============================================================
# 1.4 Confidence Interval
# ============================================================

# Sample data
sample_data = np.random.normal(loc=75, scale=12, size=40)
n = len(sample_data)
mean = sample_data.mean()
se = stats.sem(sample_data)  # standard error

# Confidence intervals at different levels
ci_levels = [0.90, 0.95, 0.99]
print('Confidence Intervals for Sample Mean')
print(f'  Sample mean: {mean:.2f}')
print(f'  Sample std: {sample_data.std():.2f}')
print(f'  Sample size: {n}')
print()

ci_results = {}
for level in ci_levels:
    ci = stats.t.interval(level, df=n-1, loc=mean, scale=se)
    ci_results[level] = ci
    print(f'  {int(level*100)}% CI: ({ci[0]:.2f}, {ci[1]:.2f})  width={ci[1]-ci[0]:.2f}')

# Visualization: CI comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confidence Intervals', fontsize=14, fontweight='bold')

# CI plot
colors = ['#3498db', '#e74c3c', '#2ecc71']
y_positions = [0, 1, 2]
for i, (level, ci) in enumerate(ci_results.items()):
    axes[0].plot([ci[0], ci[1]], [y_positions[i], y_positions[i]], '-', linewidth=4, color=colors[i],
                 label=f'{int(level*100)}% CI: ({ci[0]:.1f}, {ci[1]:.1f})')
    axes[0].plot(mean, y_positions[i], 'ko', markersize=8)

axes[0].set_yticks(y_positions)
axes[0].set_yticklabels([f'{int(l*100)}% CI' for l in ci_levels])
axes[0].axvline(mean, color='black', linestyle='--', alpha=0.5, label=f'Mean = {mean:.2f}')
axes[0].set_title('CI Width by Confidence Level')
axes[0].set_xlabel('Value')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Bootstrap CI demonstration
n_bootstrap = 500
bootstrap_means = []
ci_contains_true = 0
true_mean = 75

for _ in range(n_bootstrap):
    boot_sample = np.random.choice(sample_data, size=n, replace=True)
    boot_mean = boot_sample.mean()
    boot_se = stats.sem(boot_sample)
    ci_boot = stats.t.interval(0.95, df=n-1, loc=boot_mean, scale=boot_se)
    bootstrap_means.append(boot_mean)
    if ci_boot[0] <= true_mean <= ci_boot[1]:
        ci_contains_true += 1

axes[1].hist(bootstrap_means, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].axvline(mean, color='red', linestyle='--', linewidth=2, label=f'Our sample mean: {mean:.2f}')
axes[1].axvline(true_mean, color='green', linestyle='--', linewidth=2, label=f'True mean: {true_mean}')
axes[1].set_title(f'Bootstrap Sampling Distribution\n95% CI coverage: {ci_contains_true/n_bootstrap*100:.1f}%')
axes[1].set_xlabel('Bootstrap Sample Mean')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.5 Normality Tests

Many statistical tests assume **normality**. We test this assumption before applying them.

### Methods:
| Test | Best for | Notes |
|------|----------|-------|
| **Shapiro-Wilk** | Small to medium samples (n < 50) | Most powerful test |
| **Kolmogorov-Smirnov** | Large samples | Compares with theoretical distribution |
| **Q-Q Plot** | Visual inspection | Graphical method |

**H₀**: Data follows a normal distribution
**H₁**: Data does not follow a normal distribution

In [ ]:
# ============================================================
# 1.5 Normality Tests
# ============================================================

# Generate different distributions
normal_data = np.random.normal(loc=50, scale=10, size=50)
skewed_data = np.random.exponential(scale=10, size=50)
bimodal_data = np.concatenate([np.random.normal(40, 5, 25), np.random.normal(60, 5, 25)])

datasets = {'Normal': normal_data, 'Skewed (Exponential)': skewed_data, 'Bimodal': bimodal_data}

# Test each distribution
print('Normality Test Results:')
print(f'{"Dataset":<25} {"Shapiro-Wilk W":>15} {"Shapiro p":>12} {"KS stat":>10} {"KS p":>10} {"Normal?":>10}')
print('-' * 90)

results = {}
for name, data in datasets.items():
    sw_stat, sw_p = stats.shapiro(data)
    ks_stat, ks_p = stats.kstest((data - data.mean()) / data.std(), 'norm')
    is_normal = sw_p > 0.05 and ks_p > 0.05
    results[name] = {'sw_stat': sw_stat, 'sw_p': sw_p, 'ks_stat': ks_stat, 'ks_p': ks_p}
    print(f'{name:<25} {sw_stat:>15.4f} {sw_p:>12.4f} {ks_stat:>10.4f} {ks_p:>10.4f} {str(is_normal):>10}')

# Visualization: Q-Q plots and distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Normality Tests: Distributions and Q-Q Plots', fontsize=14, fontweight='bold')

for col_idx, (name, data) in enumerate(datasets.items()):
    # Distribution
    axes[0, col_idx].hist(data, bins=15, density=True, color='steelblue', alpha=0.7, edgecolor='black')
    x_range = np.linspace(data.min(), data.max(), 100)
    axes[0, col_idx].plot(x_range, stats.norm.pdf(x_range, data.mean(), data.std()),
                          'r-', linewidth=2, label='Normal fit')
    sw_p = results[name]['sw_p']
    axes[0, col_idx].set_title(f'{name}\nShapiro-Wilk p={sw_p:.4f}')
    axes[0, col_idx].set_xlabel('Value')
    axes[0, col_idx].set_ylabel('Density')
    axes[0, col_idx].legend(fontsize=8)
    axes[0, col_idx].grid(True, alpha=0.3)

    # Q-Q plot
    stats.probplot(data, dist='norm', plot=axes[1, col_idx])
    axes[1, col_idx].set_title(f'Q-Q Plot: {name}')
    axes[1, col_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 1.6 Levene's Test (Homogeneity of Variance)

**Levene's test** checks if multiple groups have equal variances (homoscedasticity).

This is an assumption of:
- Independent samples t-test
- ANOVA

### Formula:
$$W = \frac{(N-k)}{(k-1)} \cdot \frac{\sum_{i=1}^{k} n_i (\bar{Z}_{i.} - \bar{Z}_{..})^2}{\sum_{i=1}^{k} \sum_{j=1}^{n_i} (Z_{ij} - \bar{Z}_{i.})^2}$$

Where $Z_{ij} = |Y_{ij} - \bar{Y}_{i.}|$

**H₀**: All groups have equal variance (homoscedasticity)
**H₁**: At least one group has a different variance

In [ ]:
# ============================================================
# 1.6 Levene's Test for Homogeneity of Variance
# ============================================================

# Case 1: Equal variances
g1_equal = np.random.normal(50, 10, 30)
g2_equal = np.random.normal(55, 10, 30)
g3_equal = np.random.normal(60, 10, 30)

# Case 2: Unequal variances
g1_unequal = np.random.normal(50, 5, 30)
g2_unequal = np.random.normal(55, 15, 30)
g3_unequal = np.random.normal(60, 25, 30)

# Levene's test
lev_stat_eq, lev_p_eq = stats.levene(g1_equal, g2_equal, g3_equal)
lev_stat_uneq, lev_p_uneq = stats.levene(g1_unequal, g2_unequal, g3_unequal)

print('Levene\'s Test for Homogeneity of Variance')
print('\nCase 1 - Equal Variances:')
print(f'  Levene W = {lev_stat_eq:.4f}, p-value = {lev_p_eq:.4f}')
print(f'  Result: {"Reject H0 (unequal variances)" if lev_p_eq < 0.05 else "Fail to reject H0 (equal variances)"}')

print('\nCase 2 - Unequal Variances:')
print(f'  Levene W = {lev_stat_uneq:.4f}, p-value = {lev_p_uneq:.4f}')
print(f'  Result: {"Reject H0 (unequal variances)" if lev_p_uneq < 0.05 else "Fail to reject H0"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Levene's Test: Equal vs Unequal Variances", fontsize=14, fontweight='bold')

# Equal variances
data_eq = pd.DataFrame({'Value': np.concatenate([g1_equal, g2_equal, g3_equal]),
                         'Group': ['G1']*30 + ['G2']*30 + ['G3']*30})
data_eq.boxplot(column='Value', by='Group', ax=axes[0])
axes[0].set_title(f'Equal Variances\nLevene p={lev_p_eq:.3f}')
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Value')
axes[0].grid(True, alpha=0.3)

# Unequal variances
data_uneq = pd.DataFrame({'Value': np.concatenate([g1_unequal, g2_unequal, g3_unequal]),
                           'Group': ['G1']*30 + ['G2']*30 + ['G3']*30})
data_uneq.boxplot(column='Value', by='Group', ax=axes[1])
axes[1].set_title(f'Unequal Variances\nLevene p={lev_p_uneq:.3f}')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Value')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
# Part 2: Advanced Statistics

Advanced statistical methods for modeling relationships, comparing multiple groups, and understanding complex patterns.

## 2.1 Regression Analysis

**Regression analysis** models the relationship between a dependent variable and one or more independent variables.

### Simple Linear Regression:
$$Y = \beta_0 + \beta_1 X + \epsilon$$

### Multiple Linear Regression:
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p + \epsilon$$

### Key Metrics:
- **R²**: Coefficient of determination (proportion of variance explained)
- **Adjusted R²**: R² penalized for number of predictors
- **p-values**: Significance of each predictor
- **Residuals**: Difference between actual and predicted values

In [ ]:
# ============================================================
# 2.1 Simple Linear Regression
# ============================================================

# Generate data
X = np.random.uniform(0, 10, 100)
y = 2 * X + 3 + np.random.normal(0, 2, 100)

# OLS regression using statsmodels
X_with_const = sm.add_constant(X)
model = sm.OLS(y, X_with_const).fit()

print('Simple Linear Regression Results:')
print(model.summary())

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Simple Linear Regression', fontsize=14, fontweight='bold')

# Regression plot
x_pred = np.linspace(X.min(), X.max(), 100)
y_pred = model.predict(sm.add_constant(x_pred))

axes[0].scatter(X, y, alpha=0.6, color='steelblue', s=50, label='Data points')
axes[0].plot(x_pred, y_pred, 'r-', linewidth=2.5, label=f'Regression line\nY = {model.params[1]:.2f}X + {model.params[0]:.2f}')
axes[0].set_title(f'Regression (R² = {model.rsquared:.3f})')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
y_fitted = model.fittedvalues
residuals = model.resid
axes[1].scatter(y_fitted, residuals, alpha=0.6, color='coral', s=50)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title('Residuals vs Fitted Values')
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2.1 Multiple Linear Regression
# ============================================================

# Generate multiple features
n_samples = 200
X1 = np.random.uniform(0, 10, n_samples)
X2 = np.random.uniform(0, 5, n_samples)
X3 = np.random.uniform(-2, 2, n_samples)
y_multi = 3 * X1 + 2 * X2 - 1.5 * X3 + 5 + np.random.normal(0, 2, n_samples)

# Create DataFrame
df_reg = pd.DataFrame({'y': y_multi, 'X1': X1, 'X2': X2, 'X3': X3})

# Multiple regression
model_multi = smf.ols('y ~ X1 + X2 + X3', data=df_reg).fit()

print('Multiple Linear Regression Results:')
print(f'  R-squared: {model_multi.rsquared:.4f}')
print(f'  Adjusted R-squared: {model_multi.rsquared_adj:.4f}')
print(f'  F-statistic: {model_multi.fvalue:.4f}')
print(f'  F-statistic p-value: {model_multi.f_pvalue:.6f}')
print()
print('Coefficients:')
for var, coef, pval in zip(model_multi.params.index, model_multi.params.values, model_multi.pvalues.values):
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f'  {var:<15}: {coef:>8.4f}  (p={pval:.4f}) {sig}')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Multiple Regression: Partial Regression Plots', fontsize=14, fontweight='bold')

for idx, col in enumerate(['X1', 'X2', 'X3']):
    axes[idx].scatter(df_reg[col], df_reg['y'], alpha=0.5, color='steelblue', s=30)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('y')
    axes[idx].set_title(f'{col} (coef={model_multi.params[col]:.2f}, p={model_multi.pvalues[col]:.4f})')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.2 ANOVA (Analysis of Variance)

**ANOVA** tests whether means of three or more groups are significantly different.

### One-way ANOVA:
$$F = \frac{MS_{between}}{MS_{within}} = \frac{SS_{between}/(k-1)}{SS_{within}/(N-k)}$$

### Two-way ANOVA:
Tests main effects of two factors **and their interaction**.

### Assumptions:
1. Independence of observations
2. Normality within groups
3. Homogeneity of variance (Levene's test)

**H₀**: All group means are equal ($\mu_1 = \mu_2 = \cdots = \mu_k$)
**H₁**: At least one group mean is different

In [ ]:
# ============================================================
# 2.2 One-way ANOVA
# ============================================================

# Three treatment groups
control = np.random.normal(50, 10, 30)
treatment_a = np.random.normal(58, 10, 30)
treatment_b = np.random.normal(65, 10, 30)

# One-way ANOVA
f_stat, p_anova = stats.f_oneway(control, treatment_a, treatment_b)

print('One-way ANOVA')
print(f'  Control:     mean={control.mean():.2f}, std={control.std():.2f}')
print(f'  Treatment A: mean={treatment_a.mean():.2f}, std={treatment_a.std():.2f}')
print(f'  Treatment B: mean={treatment_b.mean():.2f}, std={treatment_b.std():.2f}')
print(f'  F-statistic: {f_stat:.4f}')
print(f'  p-value: {p_anova:.6f}')
print(f'  Result: {"Reject H0 (group means differ)" if p_anova < 0.05 else "Fail to reject H0"}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('One-way ANOVA', fontsize=14, fontweight='bold')

# Box plot
all_data = np.concatenate([control, treatment_a, treatment_b])
groups = ['Control']*30 + ['Treatment A']*30 + ['Treatment B']*30
df_anova = pd.DataFrame({'Score': all_data, 'Group': groups})
df_anova.boxplot(column='Score', by='Group', ax=axes[0])
axes[0].set_title(f'One-way ANOVA\nF={f_stat:.2f}, p={p_anova:.4f}')
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Score')
axes[0].grid(True, alpha=0.3)

# Mean plot with CI
group_means = df_anova.groupby('Group')['Score'].mean()
group_sems = df_anova.groupby('Group')['Score'].sem()
group_labels = group_means.index.tolist()

axes[1].bar(group_labels, group_means.values, yerr=group_sems.values * 1.96,
            capsize=5, color=['steelblue', 'coral', 'green'], alpha=0.7, edgecolor='black')
axes[1].set_title('Group Means with 95% CI')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Mean Score')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 2.2 Two-way ANOVA
# ============================================================

# Create two-way factorial design
np.random.seed(42)
n_per_cell = 15

# Factor A: Diet (Low, High carb)
# Factor B: Exercise (None, Moderate, High)
data_rows = []
for diet in ['Low_Carb', 'High_Carb']:
    for exercise in ['None', 'Moderate', 'High']:
        diet_effect = 5 if diet == 'Low_Carb' else 0
        exercise_effect = 0 if exercise == 'None' else (3 if exercise == 'Moderate' else 7)
        interaction = 2 if (diet == 'Low_Carb' and exercise == 'High') else 0
        values = np.random.normal(70 + diet_effect + exercise_effect + interaction, 5, n_per_cell)
        for v in values:
            data_rows.append({'Weight_Loss': v, 'Diet': diet, 'Exercise': exercise})

df_2way = pd.DataFrame(data_rows)

# Two-way ANOVA using statsmodels
model_2way = smf.ols('Weight_Loss ~ C(Diet) + C(Exercise) + C(Diet):C(Exercise)', data=df_2way).fit()
anova_table = anova_lm(model_2way, typ=2)

print('Two-way ANOVA Table:')
print(anova_table.round(4))

# Interaction plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Two-way ANOVA: Diet x Exercise', fontsize=14, fontweight='bold')

# Interaction plot
exercise_order = ['None', 'Moderate', 'High']
for diet in ['Low_Carb', 'High_Carb']:
    means = [df_2way[(df_2way['Diet']==diet) & (df_2way['Exercise']==ex)]['Weight_Loss'].mean()
             for ex in exercise_order]
    axes[0].plot(exercise_order, means, 'o-', markersize=8, linewidth=2, label=diet)

axes[0].set_title('Interaction Plot: Diet x Exercise')
axes[0].set_xlabel('Exercise Level')
axes[0].set_ylabel('Mean Weight Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Grouped box plot
df_2way.boxplot(column='Weight_Loss', by=['Diet', 'Exercise'], ax=axes[1], figsize=(8, 5))
axes[1].set_title('Distribution by Diet and Exercise')
axes[1].set_xlabel('Diet, Exercise')
axes[1].set_ylabel('Weight Loss')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.3 Correlation Analysis

**Correlation** measures the strength and direction of the linear relationship between variables.

### Types:
| Method | Data Type | Assumption | Formula |
|--------|-----------|-----------|---------|
| **Pearson** | Continuous | Normal, linear | $r = \frac{\sum(X-\bar{X})(Y-\bar{Y})}{\sqrt{\sum(X-\bar{X})^2 \sum(Y-\bar{Y})^2}}$ |
| **Spearman** | Ordinal/Rank | Monotonic | Rank-based Pearson |
| **Kendall** | Ordinal | Non-parametric | Concordant/discordant pairs |

### Interpretation:
- $|r| = 0.0 - 0.3$: Weak
- $|r| = 0.3 - 0.7$: Moderate
- $|r| = 0.7 - 1.0$: Strong

In [ ]:
# ============================================================
# 2.3 Correlation Analysis: Pearson, Spearman, Kendall
# ============================================================

# Generate correlated data
n = 100
x = np.random.normal(0, 1, n)

# Linear relationship (good for Pearson)
y_linear = 2 * x + np.random.normal(0, 0.5, n)

# Monotonic but non-linear (better for Spearman)
y_monotonic = np.exp(x * 0.7) + np.random.normal(0, 0.5, n)

# With outliers
y_outliers = 1.5 * x + np.random.normal(0, 0.5, n)
y_outliers[np.random.choice(n, 5)] = np.random.uniform(-10, 10, 5)  # add outliers

# Calculate correlations
datasets_corr = {
    'Linear': (x, y_linear),
    'Monotonic (non-linear)': (x, y_monotonic),
    'With Outliers': (x, y_outliers)
}

print('Correlation Comparison:')
print(f'{"Dataset":<25} {"Pearson":>10} {"Spearman":>10} {"Kendall":>10}')
print('-' * 60)

for name, (xi, yi) in datasets_corr.items():
    r_pearson, p_p = stats.pearsonr(xi, yi)
    r_spearman, p_s = stats.spearmanr(xi, yi)
    r_kendall, p_k = stats.kendalltau(xi, yi)
    print(f'{name:<25} {r_pearson:>10.4f} {r_spearman:>10.4f} {r_kendall:>10.4f}')

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Correlation Methods Comparison', fontsize=14, fontweight='bold')

for col_idx, (name, (xi, yi)) in enumerate(datasets_corr.items()):
    r_p, _ = stats.pearsonr(xi, yi)
    r_s, _ = stats.spearmanr(xi, yi)
    axes[col_idx].scatter(xi, yi, alpha=0.6, color='steelblue', s=40)
    axes[col_idx].set_title(f'{name}\nPearson={r_p:.3f}, Spearman={r_s:.3f}')
    axes[col_idx].set_xlabel('X')
    axes[col_idx].set_ylabel('Y')
    axes[col_idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.4 Post-hoc Tests (Multiple Comparisons)

When ANOVA shows significant differences, **post-hoc tests** identify **which specific groups differ**.

### Methods:
| Method | When to use |
|--------|------------|
| **Tukey HSD** | Equal sample sizes, conservative |
| **Bonferroni** | Very conservative, controls family-wise error |
| **Dunn's test** | Non-parametric alternative |

### Multiple Comparison Problem:
With 3 groups and α=0.05:
$$P(\text{at least one false positive}) = 1 - (1-0.05)^3 = 0.143$$

Post-hoc tests control this **family-wise error rate (FWER)**.

In [ ]:
# ============================================================
# 2.4 Post-hoc Test: Tukey HSD
# ============================================================

# Generate 4 groups with different means
g1 = np.random.normal(50, 8, 25)
g2 = np.random.normal(60, 8, 25)
g3 = np.random.normal(55, 8, 25)
g4 = np.random.normal(70, 8, 25)

all_values = np.concatenate([g1, g2, g3, g4])
group_labels = ['Group 1']*25 + ['Group 2']*25 + ['Group 3']*25 + ['Group 4']*25

# One-way ANOVA first
f_stat_ph, p_value_ph = stats.f_oneway(g1, g2, g3, g4)
print(f'ANOVA: F={f_stat_ph:.4f}, p={p_value_ph:.6f}')
print()

# Tukey HSD post-hoc test
tukey_result = pairwise_tukeyhsd(all_values, group_labels, alpha=0.05)
print('Tukey HSD Post-hoc Test Results:')
print(tukey_result)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Post-hoc Tests: Tukey HSD', fontsize=14, fontweight='bold')

# Box plot
df_posthoc = pd.DataFrame({'Score': all_values, 'Group': group_labels})
df_posthoc.boxplot(column='Score', by='Group', ax=axes[0])
axes[0].set_title(f'Group Distributions\nANOVA: F={f_stat_ph:.2f}, p={p_value_ph:.4f}')
axes[0].set_xlabel('Group')
axes[0].set_ylabel('Score')
axes[0].grid(True, alpha=0.3)

# Tukey confidence intervals
tukey_result.plot_simultaneous(ax=axes[1])
axes[1].set_title('Tukey HSD: 95% Confidence Intervals\n(Non-overlapping = significant difference)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 📝 Summary

### Part 1 - Intermediate Statistics:
| Test | Purpose | Key Metric |
|------|---------|-----------|
| T-test | Compare 2 group means | t-statistic, p-value |
| Z-test | Compare mean to population (σ known) | z-statistic, p-value |
| Chi-square | Test categorical independence | χ² statistic, p-value |
| Confidence Interval | Estimate population parameter | CI range, width |
| Shapiro-Wilk | Test normality (small n) | W statistic, p-value |
| KS test | Test normality (large n) | D statistic, p-value |
| Levene's test | Test equal variances | W statistic, p-value |

### Part 2 - Advanced Statistics:
| Method | Purpose | Key Metric |
|--------|---------|-----------|
| Linear Regression | Model continuous relationship | R², coefficients |
| Multiple Regression | Multiple predictors | Adjusted R², F-stat |
| One-way ANOVA | Compare 3+ group means | F-statistic, p-value |
| Two-way ANOVA | Two factors + interaction | F-statistics per factor |
| Pearson r | Linear correlation | r, p-value |
| Spearman ρ | Monotonic correlation | ρ, p-value |
| Kendall τ | Non-parametric correlation | τ, p-value |
| Tukey HSD | Post-hoc pairwise comparison | Adjusted p-values |